# Sentiment Analysis
This notebook is dedicated to augment textual data to evaluate the robustness of NLP models in sentiment analysis tasks. By introducing various types of noise, such as character swaps, synonym substitutions, and other perturbations, to simulate real-world scenarios where textual data may contain inconsistencies or errors. This step is critical for assessing how well models can generalize to imperfect input data and maintain classification performance.

# 1. Load Test Sets

In [2]:
import pandas as pd
from tqdm import tqdm
from textattack.augmentation import EmbeddingAugmenter, CharSwapAugmenter, EasyDataAugmenter

c:\Users\Bartek\AppData\Local\pypoetry\Cache\virtualenvs\sentiment-analysis-UYCV2lRK-py3.10\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
import sys

src_path = os.path.abspath('../src')
sys.path.append(src_path)

from preprocessing.data_utils import DataUtils

In [4]:
# load datasets
amazon_file = '../data/preprocessed/amazon_test.csv'
sentiment140_file = '../data/preprocessed/sentiment140_test.csv'
yelp_file = '../data/preprocessed/yelp_test.csv'

# read datasets
amazon_test_df = pd.read_csv(amazon_file, engine='python', encoding="ISO-8859-1")
sentiment140_test_df = pd.read_csv(sentiment140_file, engine='python', encoding="ISO-8859-1")
yelp_test_df = pd.read_csv(yelp_file, engine='python', encoding="ISO-8859-1")

# 2. Functions for Generating Test Sets
Here, functions are defined to introduce noise into the datasets using techniques like character-level swaps, embedding-based word substitutions, and simple data augmentation methods.

In [5]:
import random
import numpy as np
import textattack
from textattack.shared import utils
import tensorflow as tf

# set random seed
seed = 42
random.seed(seed)
np.random.seed(seed)
textattack.shared.utils.set_seed(seed)

# set gpu as device
utils.device = 'cuda'

## 2.1 Functions for Amazon Books Reviews and Sentiment140

In [24]:
# initialize augmenters

# swaps, insertions, deletions, or substitutions
char_swap_augmenter = CharSwapAugmenter()

# replaces words in the text with semantically similar ones
embedding_augmenter = EmbeddingAugmenter(transformations_per_example=1)

# basic augmentation techniques, including synonym replacement, random insertion, random swapping of words, and random deletion
easy_data_augmenter = EasyDataAugmenter()

[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Bartek\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [25]:
def process_batch_with_augmenter(augmenter, texts):
    return augmenter.augment_many(texts)

In [26]:
# generates and saves clean and noisy test datasets (character swaps, embedding-based, EDA) 
# to evaluate model robustness. Prints examples of each dataset for verification

def generate_noisy_datasets(output_dir, test_df, dataset_name, batch_size=1000):

    # clean test set
    clean_test_df = test_df
    print("\nExamples of clean test data:")
    print(clean_test_df.head())

    # test set with character swaps
    print("\nAdding character swap noise...")
    char_swap_reviews = []
    for i in tqdm(range(0, len(clean_test_df), batch_size), desc="CharSwapAugmenter"):
        batch = clean_test_df['Review'].iloc[i:i + batch_size].tolist()
        # Convert lists to strings if needed
        char_swap_reviews.extend([" ".join(text) if isinstance(text, list) else text for text in process_batch_with_augmenter(char_swap_augmenter, batch)])
    char_swap_df = test_df.copy()
    char_swap_df['Review'] = char_swap_reviews
    print("\nExamples of test data with character swaps:")
    print(char_swap_df.head())
    print("\nTest data with character swaps saved successfully")
    char_swap_df.to_csv(os.path.join(output_dir, f"{dataset_name}_test_char_swap.csv"), index=False, encoding='utf-8')

    # test set with embedding-based substitutions
    print("\nAdding embedding-based noise...")
    embedding_reviews = []
    for i in tqdm(range(0, len(clean_test_df), batch_size), desc="EmbeddingAugmenter"):
        batch = clean_test_df['Review'].iloc[i:i + batch_size].tolist()
        embedding_reviews.extend([" ".join(text) if isinstance(text, list) else text for text in process_batch_with_augmenter(embedding_augmenter, batch)])
    embedding_df = test_df.copy()
    embedding_df['Review'] = embedding_reviews
    print("\nExamples of test data with embedding-based noise:")
    print(embedding_df.head())
    print("\nTest data with embedding-based noise saved successfully")
    embedding_df.to_csv(os.path.join(output_dir, f"{dataset_name}_test_embedding.csv"), index=False, encoding='utf-8')

    # test set with EDA noise
    print("\nAdding EDA noise...")
    eda_reviews = []
    for i in tqdm(range(0, len(clean_test_df), batch_size), desc="EasyDataAugmenter"):
        batch = clean_test_df['Review'].iloc[i:i + batch_size].tolist()
        # EDA augmenter does not support batch processing natively
        eda_reviews.extend([" ".join(text) if isinstance(text, list) else text for text in [easy_data_augmenter.augment(text)[0] for text in batch]])
    eda_df = test_df.copy()
    eda_df['Review'] = eda_reviews
    print("\nExamples of test data with EDA noise:")
    print(eda_df.head())
    print("\nTest data with EDA noise saved successfully")
    eda_df.to_csv(os.path.join(output_dir, f"{dataset_name}_test_eda.csv"), index=False, encoding='utf-8')

    print(f"Clean and noisy test sets for {dataset_name} dataset saved successfully!")

## 2.2 Functions for Yelp Reviews

In [6]:
# set random seed
seed = 42
random.seed(seed)

In [7]:
# augment text by swapping random adjacent characters
def char_swap(text, swap_prob=0.1):
    chars = list(text)
    # return unchanged if text has 1 or fewer characters
    if len(chars) <= 1:
        return text  

    # limit the number of swaps to 10% of the number of characters
    max_swaps = max(1, len(chars) // 10)
    num_swaps = 0

    for i in range(len(chars)):
        if random.random() < swap_prob and num_swaps < max_swaps:
            if i < len(chars) - 1:
                chars[i], chars[i + 1] = chars[i + 1], chars[i]
                num_swaps += 1  

    return ''.join(chars)

In [8]:
# apply random deletion, insertion, and swap
def eda_combined_transform(text, deletion_prob=0.05, insertion_prob=0.1, eda_swap_prob=0.1):
    words = text.split()
    if len(words) <= 1:
        return text

    # calculate the number of insertions and swaps as 10% of the word count
    num_insertions = max(1, int(len(words) * insertion_prob))
    num_swaps = max(1, int(len(words) * eda_swap_prob))

    # random deletion
    words = [word for word in words if random.random() > deletion_prob]
    # ensure at least one word remains
    if len(words) == 0:
        words = [random.choice(text.split())]

    # random insertion
    for _ in range(num_insertions):
        word_to_insert = random.choice(text.split())
        insert_position = random.randint(0, len(words))
        words.insert(insert_position, word_to_insert)

    # random swap
    for _ in range(num_swaps):
        # ensure there are at least two words to swap
        if len(words) > 1:
            idx1, idx2 = random.sample(range(len(words)), 2)
            words[idx1], words[idx2] = words[idx2], words[idx1]

    return ' '.join(words)

In [9]:
# apply char_swap augmentation and save results to a file.
def augment_with_char_swap(output_file, test_df, batch_size=1000, swap_prob=0.1):
    print("\nAdding char_swap noise...")
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write('Polarity,Review\n')  
        for i in tqdm(range(0, len(test_df), batch_size), desc="Processing char_swap"):
            batch = test_df.iloc[i:i + batch_size]
            augmented_reviews = [char_swap(text, swap_prob=swap_prob) for text in batch['Review']]
            for polarity, review in zip(batch['Polarity'], augmented_reviews):
                f.write(f'{polarity},{review}\n')
    print("\nTest data with char_swap noise saved successfully")

In [10]:
# apply EDA combined transformation (random deletion, insertion, and swap) and save results to a file.
def augment_with_combined_transform(output_file, test_df, batch_size=1000, deletion_prob=0.05, insertion_prob=0.1, eda_swap_prob=0.1):
    print("\nAdding combined EDA noise (deletion, insertion, and swap)...")
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write('Polarity,Review\n')  
        for i in tqdm(range(0, len(test_df), batch_size), desc="Processing EDA Combined Transform"):
            batch = test_df.iloc[i:i + batch_size]
            augmented_reviews = [eda_combined_transform(text, deletion_prob=deletion_prob, insertion_prob=insertion_prob, eda_swap_prob=eda_swap_prob) for text in batch['Review']]
            for polarity, review in zip(batch['Polarity'], augmented_reviews):
                f.write(f'{polarity},{review}\n')
    print("\nTest data with combined EDA noise saved successfully")

In [11]:
# generate and save datasets with char_swap, combined EDA transformation and a combination of both.
def generate_noisy_datasets_simple(output_dir, test_df, dataset_name, batch_size=1000, swap_prob=0.1, deletion_prob=0.05, insertion_prob=0.1, eda_swap_prob=0.1):
    # display examples of clean test data
    print("\nExamples of clean test data:")
    print(test_df.head())

    # augmentation with char_swap
    char_swap_output_file = os.path.join(output_dir, f"{dataset_name}_test_char_swap.csv")
    augment_with_char_swap(char_swap_output_file, test_df, batch_size, swap_prob=swap_prob)
    char_swap_df = pd.read_csv(char_swap_output_file)  
    print("\nExamples of char_swap data:")
    print(char_swap_df.head())

    # augmentation with combined transform
    combined_transform_output_file = os.path.join(output_dir, f"{dataset_name}_test_transform.csv")
    augment_with_combined_transform(combined_transform_output_file, test_df, batch_size, deletion_prob=deletion_prob, insertion_prob=insertion_prob, eda_swap_prob=eda_swap_prob)
    combined_transform_df = pd.read_csv(combined_transform_output_file)  
    print("\nExamples of EDA combined transform data:")
    print(combined_transform_df.head())

    # combination of char_swap and combined transform
    char_swap_and_combined_output_file = os.path.join(output_dir, f"{dataset_name}_test_combined.csv")
    augment_with_combined_transform(char_swap_and_combined_output_file, char_swap_df, batch_size, deletion_prob=deletion_prob, insertion_prob=insertion_prob, eda_swap_prob=eda_swap_prob)
    char_swap_and_combined_df = pd.read_csv(char_swap_and_combined_output_file)  
    print("\nDataset with char_swap and combined transform:")
    print(char_swap_and_combined_df.head())

    print(f"\nAll datasets for {dataset_name} have been successfully saved!")

# 3. Generate Test Sets

In [12]:
# path to the folder where files will be saved (relative path)
output_dir = "../data/preprocessed/"

In [13]:
generate_noisy_datasets(output_dir, amazon_test_df, "amazon")


Examples of clean test data:
   Polarity                                  Review
0         1                               good book
1         1           almost unbelievable rivetting
2         0  book seems depict mother serious issue
3         1                          great book boy
4         1                       wonderous history

Adding character swap noise...


CharSwapAugmenter: 100%|██████████| 106/106 [00:59<00:00,  1.79it/s]



Examples of test data with character swaps:
   Polarity                                   Review
0         1                                 good ook
1         1             almst unbelievable rivetting
2         0  book seems depict motWher serious issue
3         1                           graet book boy
4         1                        wonderous hitsory

Test data with character swaps saved successfully

Adding embedding-based noise...


EmbeddingAugmenter: 100%|██████████| 106/106 [10:46<00:00,  6.10s/it]



Examples of test data with embedding-based noise:
   Polarity                                      Review
0         1                                  good books
1         1                circa unbelievable rivetting
2         0  book seems describing mother serious issue
3         1                          wonderful book boy
4         1                          wonderous historic

Test data with embedding-based noise saved successfully

Adding EDA noise...


EasyDataAugmenter: 100%|██████████| 106/106 [03:32<00:00,  2.00s/it]


Examples of test data with EDA noise:
   Polarity                                       Review
0         1                                    book good
1         1                unbelievable almost rivetting
2         0  limn book seems depict mother serious issue
3         1                         great playscript boy
4         1                            history wonderous

Test data with EDA noise saved successfully
Clean and noisy test sets for amazon dataset saved successfully!


In [14]:
generate_noisy_datasets(output_dir, sentiment140_test_df, "sentiment140")


Examples of clean test data:
   Polarity                                             Review
0         0  question back whn said question seems like dnt...
1         1                    hmm let try lb day could settle
2         0                                            wishing
3         0                   totally cuddling mood one cuddle
4         1            old school cha cha slide like every day

Adding character swap noise...


CharSwapAugmenter: 100%|██████████| 239/239 [09:37<00:00,  2.42s/it]



Examples of test data with character swaps:
   Polarity                                             Review
0         0  question back whn sai question seems like dnt ...
1         1                   hmm ldet try lb day could settle
2         0                                           wisShing
3         0                  totally cuddling mEood one cuddle
4         1             old school cha cha slide ike every day

Test data with character swaps saved successfully

Adding embedding-based noise...


EmbeddingAugmenter: 100%|██████████| 239/239 [1:47:08<00:00, 26.90s/it]



Examples of test data with embedding-based noise:
   Polarity                                             Review
0         0  question back whn said matter seems like dnt c...
1         1                   hmm let try lb day could settles
2         0                                             desire
3         0                   utterly cuddling mood one cuddle
4         1        antigua school cha cha slide like every day

Test data with embedding-based noise saved successfully

Adding EDA noise...


EasyDataAugmenter: 100%|██████████| 239/239 [33:23<00:00,  8.38s/it]



Examples of test data with EDA noise:
   Polarity                                             Review
0         0  indorse question back whn said question seems ...
1         1                    hmm let try day lb could settle
2         0                                            wishing
3         0               totally cuddling modality one cuddle
4         1   old civilise school cha cha slide like every day

Test data with EDA noise saved successfully
Clean and noisy test sets for sentiment140 dataset saved successfully!


In [13]:
# due to complexity and the number of tokens per record, the hype was changed for yelp test set
generate_noisy_datasets_simple(output_dir, yelp_test_df, "yelp")


Examples of clean test data:
   Polarity                                             Review
0         1  found place heard great buffet feast buffet en...
1         0  horrible experience walmart auto center hour w...
2         1  ive heard place year never gone today parked g...
3         0  gift certificate otherwise would never gone me...
4         0  watch rip brought back full tank still got fue...

Adding char_swap noise...


Processing char_swap: 100%|██████████| 84/84 [00:03<00:00, 27.07it/s]



Test data with char_swap noise saved successfully

Examples of char_swap data:
   Polarity                                             Review
0         1  fuond palec ehard gerat bufeft feast buffte ne...
1         0  horirble experience wlamrat aut coenter hour w...
2         1  ive heard lpace eyar never gone today parked g...
3         0  gift certifciateo therwsie woudl never onge me...
4         0  wacthr pi broughtb ack ufll tan ktsillg ot fue...

Adding combined EDA noise (deletion, insertion, and swap)...


Processing EDA Combined Transform: 100%|██████████| 84/84 [00:04<00:00, 20.14it/s]



Test data with combined EDA noise saved successfully

Examples of EDA combined transform data:
   Polarity                                             Review
0         1  found place heard gamble machine buffet feast ...
1         0  horrible needless walmart auto center hour stu...
2         1  looked heard place year gone today az walked e...
3         0  gift certificate otherwise salty never gone wo...
4         0  watch rip brought back tank smelled still got ...

Adding combined EDA noise (deletion, insertion, and swap)...


Processing EDA Combined Transform: 100%|██████████| 84/84 [00:04<00:00, 20.34it/s]



Test data with combined EDA noise saved successfully

Dataset with char_swap and combined transform:
   Polarity                                             Review
0         1  fuond palec dont ehard gerat bufeft feast buff...
1         0  oilm cahnge wlamrat aut enver hour waiting exp...
2         1  ive heard lpace szie told never gone today par...
3         0  gift certifciateo therwsie woudl onge meat eat...
4         0  dya garbaeg wacthr ride bill ack ufll tan brou...

All datasets for yelp have been successfully saved!
